In [1]:
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader, random_split

In [2]:
df = pd.read_parquet("data/train.parquet")

In [4]:
df.head()

,feature0,feature1,feature2,feature3,feature4,feature5,feature6,feature7,feature8,feature9,...,feature17,feature18,feature19,feature20,feature21,feature22,feature23,feature24,target0,target1
0,32.910908,17.376350,77.557840,2.929855,gas1,20.487150,57.633085,49.245392,-44.124381,23.400064,...,14.203652,9.689942,17.951627,24.198589,102.448710,110.529868,56.817260,12.887802,27.050891,6.502743
1,41.263782,22.419445,47.945514,-25.847472,gas2,21.461239,3.474080,49.659980,-13.553188,-13.047593,...,-14.451904,-0.906120,105.724742,-9.435222,-16.060150,37.286110,61.224272,39.822424,84.127890,76.578716
2,25.580283,17.376350,77.654180,2.799411,gas1,20.487150,28.938295,49.245392,-44.124381,20.610679,...,14.395054,9.779019,17.951638,24.999453,101.728213,111.471534,52.664304,12.887802,22.080133,3.036043
3,33.756900,17.376350,73.049625,2.953982,gas1,20.487150,28.932311,49.245392,-44.124381,18.107963,...,14.184314,9.798969,17.951675,23.990300,101.312113,115.589451,56.840719,12.887802,30.234082,8.910795
4,4.223732,38.772534,48.015553,-25.843943,gas2,24.635721,12.011581,51.030938,84.244199,-17.735680,...,-14.439953,8.262354,90.187207,-2.901661,-16.060150,37.398779,40.488468,128.295838,71.128092,50.475082


In [5]:
feature_cols = [col for col in df.columns if col.startswith('feature') and col != 'feature4']

X = df[feature_cols].values.astype(np.float32)
y0 = df['target0'].values.astype(np.float32)
y1 = df['target1'].values.astype(np.float32)

In [6]:
class GasDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y
        
    def __len__(self):
        return len(self.y)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [7]:
train_size = 0.5
test_size = 0.5

dataset0 = GasDataset(X, y0)
dataset1 = GasDataset(X, y1)

train_dataset0, test_dataset0 = random_split(dataset0, [train_size, test_size])
train_dataset1, test_dataset1 = random_split(dataset1, [train_size, test_size])

In [8]:
batch_size = 64

train_loader0 = DataLoader(train_dataset0, batch_size=batch_size, shuffle=True)
test_loader0 = DataLoader(test_dataset0, batch_size=batch_size, shuffle=False)

train_loader1 = DataLoader(train_dataset1, batch_size=batch_size, shuffle=True)
test_loader1 = DataLoader(test_dataset1, batch_size=batch_size, shuffle=False)

In [9]:
def GammaLoss(output, target):
    loss = target / output + torch.log(output)
    return loss.mean()

In [10]:
class BranchBlock(nn.Module):
    def __init__(self, input_dim):
        super(BranchBlock, self).__init__()
        self.block = nn.Sequential(
            nn.Linear(in_features=input_dim, out_features=40),
            nn.Tanh(),
            nn.Linear(in_features=40, out_features=20)
        )
    
    def forward(self, x):
        x = self.block(x)
        return torch.exp(x)

In [11]:
class BranchingNetwork(nn.Module):
    def __init__(self, input_dim):
        super(BranchingNetwork, self).__init__()
        self.branches = nn.ModuleList([
            BranchBlock(input_dim),
            BranchBlock(input_dim)
        ])
        self.output_layer = nn.Linear(in_features=40, out_features=1)
        
    def forward(self, x):
        branch_outputs = [branch(x) for branch in self.branches]
        combined = torch.cat(branch_outputs, dim=1)
        return self.output_layer(combined)

In [12]:
input_dim = len(feature_cols)

model0 = BranchingNetwork(input_dim)
model1 = BranchingNetwork(input_dim)

optimizer0 = torch.optim.Adam(model0.parameters(), lr=0.001)
optimizer1 = torch.optim.Adam(model1.parameters(), lr=0.001)

n_epochs = 100

In [13]:
for epoch in range(n_epochs):
    losses = 0
    for x_batch, y_batch in train_loader0:
        optimizer0.zero_grad()
        pred = model0(x_batch)
        loss = GammaLoss(pred, y_batch)
        losses += loss.item() / len(train_loader0)
        loss.backward()
        optimizer0.step()
    
    if (epoch + 1) % 10 == 0:
        print(f'epoch: {epoch + 1}  loss: {losses:.3f}')

epoch: 10  loss: 4.832
epoch: 20  loss: 4.832
epoch: 30  loss: 4.832
epoch: 40  loss: 4.832
epoch: 50  loss: 4.832
epoch: 60  loss: 4.832
epoch: 70  loss: 4.832
epoch: 80  loss: 4.832
epoch: 90  loss: 4.832
epoch: 100  loss: 4.832


In [14]:
for epoch in range(n_epochs):
    losses = 0
    for x_batch, y_batch in train_loader1:
        optimizer1.zero_grad()
        pred = model1(x_batch)
        loss = GammaLoss(pred, y_batch)
        losses += loss.item() / len(train_loader1)
        loss.backward()
        optimizer1.step()
    
    if (epoch + 1) % 10 == 0:
        print(f'epoch: {epoch + 1}  loss: {losses:.3f}')

epoch: 10  loss: 4.100
epoch: 20  loss: 4.100
epoch: 30  loss: 4.100
epoch: 40  loss: 4.100
epoch: 50  loss: 4.100
epoch: 60  loss: 4.100
epoch: 70  loss: 4.100
epoch: 80  loss: 4.100
epoch: 90  loss: 4.100
epoch: 100  loss: 4.100


In [15]:
def calculate_mape(model, loader):
    model.eval()
    total_mape = 0
    count = 0
    with torch.no_grad():
        for x_batch, y_batch in loader:
            pred = model(x_batch)
            mape = torch.mean(torch.abs((y_batch - pred) / y_batch))
            total_mape += mape.item() * len(y_batch)
            count += len(y_batch)
    return total_mape / count

In [16]:
mape0 = calculate_mape(model0, test_loader0)
mape1 = calculate_mape(model1, test_loader1)

print(f"MAPE для target0: {mape0:.2f}")
print(f"MAPE для target1: {mape1:.2f}")

MAPE для target0: 0.75
MAPE для target1: 3.37
